# Stage B: route-bank and router training

Load the verified fixed-adapter handoff from Notebook 01a. Train independent route experts, select a static-bank action, train the router, then export `ars_training_bundle.zip` for evaluation notebooks.

## Required input and runtime

Attach either the `ars_fixed_adapter_bundle.zip` from Notebook 01a or its extracted folder. For an extracted Kaggle Model input, the folder root must contain `manifest.json` alongside `checkpoints/`, `calibration/`, `training/`, and `code/`. This continuation validates the manifest, artifact and rollout hashes, and pinned repository commit. It trains the candidate route bank and router; it does not recollect rollouts or retrain the fixed-route adapter.


In [ ]:
import hashlib, json, os, shutil, subprocess, sys, zipfile
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

RUNTIME_PLATFORM = "auto"
REPO_URL = "https://github.com/Abdullah182155/adaptive-residual-steering.git"
FIXED_ADAPTER_BUNDLE_OVERRIDE = None  # Set to the exact ZIP path or extracted handoff folder when needed.
if RUNTIME_PLATFORM == "auto":
    RUNTIME_PLATFORM = "kaggle" if Path("/kaggle/working").is_dir() else "colab" if Path("/content").is_dir() else "unknown"
if RUNTIME_PLATFORM == "kaggle":
    WORK_ROOT, INPUT_ROOT = Path("/kaggle/working"), Path("/kaggle/input")
elif RUNTIME_PLATFORM == "colab":
    WORK_ROOT, INPUT_ROOT = Path("/content"), Path("/content")
else:
    raise RuntimeError("Could not detect Kaggle or Colab")

EXPECTED_HANDOFF_FORMAT = "ars_fixed_adapter_handoff_v1"
EXPECTED_ARCHITECTURE = "ars.fixed_route_low_rank_residual"
EXPECTED_ARCHITECTURE_VERSION = "1.0.0"

def _read_manifest(path):
    try:
        return json.loads(Path(path).read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError, UnicodeDecodeError):
        return None

def _is_handoff_manifest(manifest):
    return (
        isinstance(manifest, dict)
        and manifest.get("format") == EXPECTED_HANDOFF_FORMAT
        and manifest.get("fixed_adapter_ready") is True
    )

def _validate_manifest(manifest):
    if not _is_handoff_manifest(manifest):
        status = manifest.get("required_phase_status") if isinstance(manifest, dict) else None
        raise ValueError(f"Input is not a complete fixed-adapter handoff: {status}")
    if (
        manifest.get("architecture_id") != EXPECTED_ARCHITECTURE
        or manifest.get("architecture_version") != EXPECTED_ARCHITECTURE_VERSION
    ):
        raise ValueError("Fixed-adapter handoff uses an unsupported architecture contract")
    raw_expected = {}
    for section in ("artifacts", "rollouts", "collection_metadata_hashes", "code"):
        entries = manifest.get(section, {})
        if not isinstance(entries, dict):
            raise ValueError(f"Invalid {section} section in fixed-adapter manifest")
        raw_expected.update(entries)
    if not raw_expected:
        raise ValueError("Fixed-adapter handoff manifest contains no files to verify")

    # Most manifest sections store {"sha256": ...}; collection metadata stores the hash string directly.
    expected = {}
    for member, metadata in raw_expected.items():
        if not isinstance(member, str):
            raise ValueError("Invalid non-string path in fixed-adapter manifest")
        relative = PurePosixPath(member)
        checksum = metadata if isinstance(metadata, str) else metadata.get("sha256") if isinstance(metadata, dict) else None
        if (
            relative.is_absolute()
            or ".." in relative.parts
            or not isinstance(checksum, str)
            or len(checksum) != 64
            or any(character not in "0123456789abcdefABCDEF" for character in checksum)
        ):
            raise ValueError(f"Invalid path or checksum in fixed-adapter manifest: {member}")
        expected[member] = {"sha256": checksum.lower()}
    return expected

def _verify_directory(source, expected):
    root = source.resolve()
    for member, metadata in expected.items():
        path = (root / Path(*PurePosixPath(member).parts)).resolve()
        if path != root and root not in path.parents:
            raise ValueError(f"Unsafe path in fixed-adapter manifest: {member}")
        if not path.is_file() or sha256_file(path) != metadata["sha256"]:
            raise ValueError(f"Missing or checksum-mismatched handoff file: {member}")

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

if FIXED_ADAPTER_BUNDLE_OVERRIDE:
    FIXED_ADAPTER_INPUT = Path(FIXED_ADAPTER_BUNDLE_OVERRIDE).expanduser()
    if not FIXED_ADAPTER_INPUT.exists():
        raise FileNotFoundError(f"Fixed-adapter input does not exist: {FIXED_ADAPTER_INPUT}")
    if FIXED_ADAPTER_INPUT.is_dir():
        HANDOFF_INPUT_TYPE = "extracted_directory"
    elif FIXED_ADAPTER_INPUT.is_file() and FIXED_ADAPTER_INPUT.suffix.lower() == ".zip":
        HANDOFF_INPUT_TYPE = "zip"
    else:
        raise ValueError("FIXED_ADAPTER_BUNDLE_OVERRIDE must point to the handoff ZIP or its extracted root folder")
else:
    zip_candidates = {
        path.resolve()
        for path in INPUT_ROOT.rglob("ars_fixed_adapter_bundle.zip")
        if path.is_file()
    }
    directory_candidates = set()
    for manifest_path in INPUT_ROOT.rglob("manifest.json"):
        candidate_manifest = _read_manifest(manifest_path)
        if _is_handoff_manifest(candidate_manifest):
            directory_candidates.add(manifest_path.parent.resolve())
    discovered = [("zip", path) for path in sorted(zip_candidates)]
    discovered += [("extracted_directory", path) for path in sorted(directory_candidates)]
    if len(discovered) != 1:
        choices = [f"{kind}: {path}" for kind, path in discovered]
        raise FileNotFoundError(
            "Expected exactly one fixed-adapter handoff, as a ZIP or an extracted folder containing "
            f"manifest.json; found: {choices}. Set FIXED_ADAPTER_BUNDLE_OVERRIDE to the exact ZIP or folder path."
        )
    HANDOFF_INPUT_TYPE, FIXED_ADAPTER_INPUT = discovered[0]

if HANDOFF_INPUT_TYPE == "extracted_directory":
    HANDOFF_MANIFEST = _read_manifest(FIXED_ADAPTER_INPUT / "manifest.json")
    EXPECTED_HANDOFF_FILES = _validate_manifest(HANDOFF_MANIFEST)
    _verify_directory(FIXED_ADAPTER_INPUT, EXPECTED_HANDOFF_FILES)
else:
    with zipfile.ZipFile(FIXED_ADAPTER_INPUT) as archive:
        names = set(archive.namelist())
        if "manifest.json" not in names:
            raise ValueError("Fixed-adapter handoff ZIP has no root manifest.json")
        HANDOFF_MANIFEST = json.loads(archive.read("manifest.json"))
        EXPECTED_HANDOFF_FILES = _validate_manifest(HANDOFF_MANIFEST)
        for member, metadata in EXPECTED_HANDOFF_FILES.items():
            if member not in names or sha256_bytes(archive.read(member)) != metadata["sha256"]:
                raise ValueError(f"Missing or checksum-mismatched ZIP member: {member}")
        for member in archive.infolist():
            relative = PurePosixPath(member.filename)
            if relative.is_absolute() or ".." in relative.parts:
                raise ValueError(f"Unsafe ZIP path: {member.filename}")

SOURCE_RUN_ID = HANDOFF_MANIFEST["run_id"]
CONTINUATION_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
base = (WORK_ROOT / "ars_session3_stage_b" / f"{SOURCE_RUN_ID}_{CONTINUATION_ID}").resolve()
base.mkdir(parents=True, exist_ok=True)
if HANDOFF_INPUT_TYPE == "extracted_directory":
    shutil.copytree(FIXED_ADAPTER_INPUT, base, dirs_exist_ok=True)
else:
    with zipfile.ZipFile(FIXED_ADAPTER_INPUT) as archive:
        archive.extractall(base)

_content_digest = hashlib.sha256()
_content_digest.update(json.dumps(HANDOFF_MANIFEST, sort_keys=True, separators=(",", ":")).encode("utf-8"))
for _member, _metadata in sorted(EXPECTED_HANDOFF_FILES.items()):
    _content_digest.update(b"\0")
    _content_digest.update(_member.encode("utf-8"))
    _content_digest.update(b"\0")
    _content_digest.update(_metadata["sha256"].encode("ascii"))
HANDOFF_CONTENT_SHA256 = _content_digest.hexdigest()

REPO_DIR = WORK_ROOT / "adaptive-residual-steering-router"
commit = HANDOFF_MANIFEST["code_commit"]
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "development"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", "development", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", commit], check=True)
sys.path.insert(0, str(REPO_DIR))
for relative, metadata in HANDOFF_MANIFEST.get("code", {}).items():
    repo_file = REPO_DIR / Path(relative).relative_to("code")
    if not repo_file.is_file() or sha256_file(repo_file) != metadata["sha256"]:
        raise ValueError(f"Repository code does not match the checkpoint snapshot: {relative}")

RUN_DIR = base
CHECKPOINT = RUN_DIR / "checkpoints/fixed_route_adapter.pt"
BANK_CHECKPOINT = RUN_DIR / "checkpoints/session3_steering_bank.pt"
NORM_FILE = RUN_DIR / "calibration/fixed_route_p99_norms.json"
CANDIDATE_NORMS = RUN_DIR / "calibration/candidate_norms_p99.json"
RFT_DIR = RUN_DIR / "training/rollouts"
ROLLOUT_FILES = [RFT_DIR / "shard0.jsonl", RFT_DIR / "shard1.jsonl"]
ROUTER_CHECKPOINT = RUN_DIR / "checkpoints/session3_router.pt"
STATIC_ACTION = RUN_DIR / "session3_static_bank_action.json"
ROUTER_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
STATIC_ACTION.parent.mkdir(parents=True, exist_ok=True)
CANDIDATE_LAYERS = list(HANDOFF_MANIFEST["candidate_layers"])
FIXED_ROUTE = list(HANDOFF_MANIFEST["route"])
RUN_ID = f"{SOURCE_RUN_ID}_bank_router_{CONTINUATION_ID}"
PHASE_STATUS = dict(HANDOFF_MANIFEST.get("phase_status", {}))
PHASE_STATUS["steering_bank"] = "pending"
PHASE_STATUS["router"] = "pending"
print({"runtime":RUNTIME_PLATFORM,"input_handoff":str(FIXED_ADAPTER_INPUT),"input_type":HANDOFF_INPUT_TYPE,"source_run":RUN_ID,"pinned_commit":commit,"training_stages":["steering_bank","router"],"rollout_files":[p.is_file() for p in ROLLOUT_FILES]},flush=True)

## Route-bank and router configuration

Notebook 01b consumes the verified Stage A handoff. The active full-run profile is `extended_quality`: 2 bank epochs, 768 examples per expert, 24 held-out screening rows, route caps of 8/8/8, 3 router epochs, and up to 2,048 router rows. Alpha calibration uses 400 questions and gains `[1.0, 1.25, 1.5, 1.75, 2.0]`. Set `SMOKE_TEST = True` only to validate the handoff and stages on a small workload.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select a CUDA GPU before running route-bank and router training")
GPU_COUNT = torch.cuda.device_count()
if not all(p.is_file() and p.stat().st_size for p in ROLLOUT_FILES):
    raise FileNotFoundError(f"The fixed-adapter handoff is missing rollout shards: {[str(p) for p in ROLLOUT_FILES]}")
for required_path in (CHECKPOINT, NORM_FILE, CANDIDATE_NORMS):
    if not required_path.is_file() or not required_path.stat().st_size:
        raise FileNotFoundError(f"The fixed-adapter handoff is missing a required artifact: {required_path}")

SMOKE_TEST = False
TRAINING_PROFILE = "extended_quality"  # Use "standard" only when a shorter exploratory run is intended.
if TRAINING_PROFILE not in {"standard", "extended_quality"}:
    raise ValueError(f"Unknown TRAINING_PROFILE: {TRAINING_PROFILE}")
if SMOKE_TEST:
    BANK_EPOCHS, ROUTE_EXAMPLES_PER_EXPERT, SCREEN_VALIDATION_ROWS = 1, 8, 4
    ALPHA_CALIB_N = 24
    MAX_PAIR_ROUTES, MAX_TRIPLE_ROUTES, MAX_QUAD_ROUTES = 2, 1, 1
    ROUTER_EPOCHS, MAX_ROUTER_ROWS = 1, 12
    STATIC_SCREEN_QUESTIONS, TASK_VALIDATION_ROWS = 8, 8
elif TRAINING_PROFILE == "extended_quality":
    # Quality-first preset: more examples per independent expert, a larger held-out
    # screen, and a larger router teacher/validation set. The router restores its
    # best held-out task-accuracy checkpoint rather than its final epoch.
    BANK_EPOCHS, ROUTE_EXAMPLES_PER_EXPERT, SCREEN_VALIDATION_ROWS = 2, 768, 24
    ALPHA_CALIB_N = 400
    MAX_PAIR_ROUTES, MAX_TRIPLE_ROUTES, MAX_QUAD_ROUTES = 8, 8, 8
    ROUTER_EPOCHS, MAX_ROUTER_ROWS = 3, 2048
    STATIC_SCREEN_QUESTIONS, TASK_VALIDATION_ROWS = 128, 84
else:
    BANK_EPOCHS, ROUTE_EXAMPLES_PER_EXPERT, SCREEN_VALIDATION_ROWS = 2, 256, 24
    ALPHA_CALIB_N = 200
    MAX_PAIR_ROUTES, MAX_TRIPLE_ROUTES, MAX_QUAD_ROUTES = 8, 6, 4
    ROUTER_EPOCHS, MAX_ROUTER_ROWS = 3, 384
    STATIC_SCREEN_QUESTIONS, TASK_VALIDATION_ROWS = 48, 32
BANK_MAX_QUESTIONS = int(HANDOFF_MANIFEST["collection_policy"]["questions"])
MAX_ROUTER_QUESTIONS = min(2000, BANK_MAX_QUESTIONS)
ROUTE_COST = 0.01
MIN_LAYER_GAP, CONTROL_GAP, SPACING_CONTROL_LIMIT = 3, 2, 8
BANK_PREFERENCE_WEIGHT = 0.25
STATIC_SCREEN_ALPHAS = [1.0, 1.5, 2.0]
# Keep calibration and router gains on the same supported domain.
ROUTER_ALPHAS = [1.0, 1.25, 1.5, 1.75, 2.0]
ALPHA_GRID = list(ROUTER_ALPHAS)
ALPHA_REPORT = RUN_DIR / "alpha_calibration.json"
CANDIDATE_LAYERS = list(HANDOFF_MANIFEST["candidate_layers"])
TRAINING_PROFILE_CONFIG = {
    "name": "smoke" if SMOKE_TEST else TRAINING_PROFILE,
    "bank_epochs": BANK_EPOCHS,
    "route_examples_per_expert": ROUTE_EXAMPLES_PER_EXPERT,
    "screen_validation_rows": SCREEN_VALIDATION_ROWS,
    "alpha_calibration_questions": ALPHA_CALIB_N,
    "router_epochs": ROUTER_EPOCHS,
    "max_router_rows": MAX_ROUTER_ROWS,
    "static_screen_questions": STATIC_SCREEN_QUESTIONS,
    "task_validation_rows": TASK_VALIDATION_ROWS,
    "router_alphas": ROUTER_ALPHAS,
    "route_cost": ROUTE_COST,
    "bank_preference_weight": BANK_PREFERENCE_WEIGHT,
    "static_screen_alphas": STATIC_SCREEN_ALPHAS,
    "max_router_questions": MAX_ROUTER_QUESTIONS,
    "max_pair_routes": MAX_PAIR_ROUTES,
    "max_triple_routes": MAX_TRIPLE_ROUTES,
    "max_quad_routes": MAX_QUAD_ROUTES,
    "min_layer_gap": MIN_LAYER_GAP,
    "control_gap": CONTROL_GAP,
    "spacing_control_limit": SPACING_CONTROL_LIMIT,
}
PHASE_STATUS["runtime_stage_b"] = "passed"
print({"visible_gpus":GPU_COUNT,"smoke_test":SMOKE_TEST,"training_profile":TRAINING_PROFILE_CONFIG,"route_caps":[MAX_PAIR_ROUTES,MAX_TRIPLE_ROUTES,MAX_QUAD_ROUTES],"candidate_layers":CANDIDATE_LAYERS},flush=True)

## Train the route bank and router

On a dual-GPU runtime, held-out alpha calibration and route-bank training run concurrently on separate GPUs. On a single-GPU runtime they run sequentially. Router training starts only after the bank checkpoint has passed its save check.

In [ ]:
from scripts.research_notebook_utils import run_live
import os

action_paths = (BANK_CHECKPOINT, ROUTER_CHECKPOINT, STATIC_ACTION)
for path in action_paths:
    path.parent.mkdir(parents=True, exist_ok=True)
(RUN_DIR / "logs").mkdir(parents=True, exist_ok=True)
base_env = os.environ.copy()
base_env["ARS_ATTENTION_IMPLEMENTATION"] = "sdpa"

alpha_command = [
    sys.executable, "-u", "scripts/run_rft_pipeline.py",
    "--stage", "alpha_sweep",
    "--steering_architecture", "fixed_route_low_rank_residual",
    "--norm_calibration", str(NORM_FILE),
    "--checkpoint", str(CHECKPOINT),
    "--n_questions", str(ALPHA_CALIB_N),
    "--alpha_values", *map(str, ALPHA_GRID),
    "--output_dir", str(RUN_DIR),
]
bank_command = [
    sys.executable, "-u", "scripts/run_session3_router.py",
    "--stage", "train_bank",
    "--output_dir", str(RUN_DIR),
    "--candidate_layers", *map(str, CANDIDATE_LAYERS),
    "--norm_calibration", str(CANDIDATE_NORMS),
    "--fixed_adapter_norm_calibration", str(NORM_FILE),
    "--fixed_adapter_checkpoint", str(CHECKPOINT),
    "--bank_checkpoint", str(BANK_CHECKPOINT),
    "--router_checkpoint", str(ROUTER_CHECKPOINT),
    "--rollout_files", *map(str, ROLLOUT_FILES),
    "--epochs", str(BANK_EPOCHS),
    "--max_questions", str(BANK_MAX_QUESTIONS),
    "--max_rows_per_question", str(HANDOFF_MANIFEST["collection_policy"]["max_training_rows_per_question"]),
    "--route_examples_per_expert", str(ROUTE_EXAMPLES_PER_EXPERT),
    "--screen_validation_rows", str(SCREEN_VALIDATION_ROWS),
    "--preference_weight", str(BANK_PREFERENCE_WEIGHT),
    "--max_pair_routes", str(MAX_PAIR_ROUTES),
    "--max_triple_routes", str(MAX_TRIPLE_ROUTES),
    "--max_quad_routes", str(MAX_QUAD_ROUTES),
    "--min_layer_gap", str(MIN_LAYER_GAP),
    "--control_gap", str(CONTROL_GAP),
    "--spacing_control_limit", str(SPACING_CONTROL_LIMIT),
    "--diversify_prompts",
]

if GPU_COUNT >= 2:
    from scripts.research_notebook_utils import run_parallel_live
    alpha_env = base_env.copy()
    alpha_env["CUDA_VISIBLE_DEVICES"] = "1"
    bank_env = base_env.copy()
    bank_env["CUDA_VISIBLE_DEVICES"] = "0"
    print("Running independent alpha calibration and bank training concurrently on separate GPUs.", flush=True)
    try:
        stage_results = dict(run_parallel_live(
            [
                ("alpha_calibration", alpha_command, alpha_env),
                ("bank_training", bank_command, bank_env),
            ],
            log_dir=RUN_DIR / "logs",
            cwd=REPO_DIR,
        ))
        PHASE_STATUS["alpha_calibration"] = (
            "passed" if stage_results.get("alpha_calibration") == 0 and ALPHA_REPORT.is_file()
            else f"failed: exit_code={stage_results.get('alpha_calibration')}, report_exists={ALPHA_REPORT.is_file()}"
        )
        PHASE_STATUS["steering_bank"] = (
            "passed" if stage_results.get("bank_training") == 0 and BANK_CHECKPOINT.is_file() and BANK_CHECKPOINT.stat().st_size
            else f"failed: exit_code={stage_results.get('bank_training')}, checkpoint_exists={BANK_CHECKPOINT.is_file()}"
        )
    except Exception as exc:
        PHASE_STATUS["alpha_calibration"] = f"failed: {type(exc).__name__}: {exc}"
        PHASE_STATUS["steering_bank"] = f"failed: {type(exc).__name__}: {exc}"
else:
    base_env["CUDA_VISIBLE_DEVICES"] = "0"
    print("One GPU detected; running alpha calibration and bank training sequentially.", flush=True)
    try:
        alpha_code = run_live(alpha_command, cwd=REPO_DIR, log_path=RUN_DIR / "logs/alpha_calibration.log", env=base_env)
        PHASE_STATUS["alpha_calibration"] = "passed" if alpha_code == 0 and ALPHA_REPORT.is_file() else f"failed: exit_code={alpha_code}, report_exists={ALPHA_REPORT.is_file()}"
    except Exception as exc:
        PHASE_STATUS["alpha_calibration"] = f"failed: {type(exc).__name__}: {exc}"
    try:
        bank_code = run_live(bank_command, cwd=REPO_DIR, log_path=RUN_DIR / "logs/bank_training.log", env=base_env)
        PHASE_STATUS["steering_bank"] = "passed" if bank_code == 0 and BANK_CHECKPOINT.is_file() and BANK_CHECKPOINT.stat().st_size else f"failed: exit_code={bank_code}, checkpoint_exists={BANK_CHECKPOINT.is_file()}"
    except Exception as exc:
        PHASE_STATUS["steering_bank"] = f"failed: {type(exc).__name__}: {exc}"

SELECTED_ALPHA = 1.0
ALPHA_SELECTION = {"method": "0.7 * 0-shot paired delta + 0.3 * 2-shot paired delta"}
if ALPHA_REPORT.is_file():
    alpha_data = json.loads(ALPHA_REPORT.read_text(encoding="utf-8"))
    alpha_scores = {}
    for shot, values in alpha_data["shots"].items():
        for item in values["candidates"]:
            alpha_scores.setdefault(float(item["alpha"]), {})[str(shot)] = float(item["paired_delta_lenient_pp"])
    eligible = [alpha for alpha, scores in alpha_scores.items() if "0" in scores and "2" in scores]
    if eligible:
        SELECTED_ALPHA = max(eligible, key=lambda alpha: (0.7 * alpha_scores[alpha]["0"] + 0.3 * alpha_scores[alpha]["2"], -abs(alpha - 1.0)))
        ALPHA_SELECTION["validation_paired_deltas"] = {str(alpha): alpha_scores[alpha] for alpha in eligible}
        ALPHA_SELECTION["selected_alpha"] = SELECTED_ALPHA
print({"alpha_status":PHASE_STATUS["alpha_calibration"],"selected_alpha":SELECTED_ALPHA,"bank_status":PHASE_STATUS["steering_bank"]},flush=True)

if PHASE_STATUS["steering_bank"] == "passed":
    router_command = [
        sys.executable, "-u", "scripts/run_session3_router.py",
        "--stage", "train_router",
        "--output_dir", str(RUN_DIR),
        "--candidate_layers", *map(str, CANDIDATE_LAYERS),
        "--norm_calibration", str(CANDIDATE_NORMS),
        "--fixed_adapter_norm_calibration", str(NORM_FILE),
        "--fixed_adapter_checkpoint", str(CHECKPOINT),
        "--bank_checkpoint", str(BANK_CHECKPOINT),
        "--router_checkpoint", str(ROUTER_CHECKPOINT),
        "--rollout_files", *map(str, ROLLOUT_FILES),
        "--epochs", str(ROUTER_EPOCHS),
        "--max_questions", str(MAX_ROUTER_QUESTIONS),
        "--max_router_rows", str(MAX_ROUTER_ROWS),
        "--route_cost", str(ROUTE_COST),
        "--router_alphas", *map(str, ROUTER_ALPHAS),
        "--static_screen_questions", str(STATIC_SCREEN_QUESTIONS),
        "--task_validation_rows", str(TASK_VALIDATION_ROWS),
        "--diversify_prompts",
    ]
    try:
        router_env = base_env.copy()
        router_env["CUDA_VISIBLE_DEVICES"] = "0"
        router_code = run_live(router_command, cwd=REPO_DIR, log_path=RUN_DIR / "logs/router_training.log", env=router_env)
        PHASE_STATUS["router"] = "passed" if router_code == 0 and ROUTER_CHECKPOINT.is_file() and STATIC_ACTION.is_file() else f"failed: exit_code={router_code}, checkpoint_exists={ROUTER_CHECKPOINT.is_file()}, static_action_exists={STATIC_ACTION.is_file()}"
    except Exception as exc:
        PHASE_STATUS["router"] = f"failed: {type(exc).__name__}: {exc}"
else:
    PHASE_STATUS["router"] = "skipped_prerequisite_failure"
print({"router_status":PHASE_STATUS["router"],"router_checkpoint":ROUTER_CHECKPOINT.is_file(),"static_action":STATIC_ACTION.is_file()},flush=True)

## Package the final training bundle

Evaluation notebooks accept only `ars_training_bundle.zip` with successful route-bank and router stages and valid artifact hashes.

In [ ]:
import hashlib
import json
import math
import torch
import zipfile
from datetime import datetime, timezone

from src.steering.route_bank import normalize_route_catalog, validate_route_expert_states


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


required = {
    "checkpoints/fixed_route_adapter.pt": CHECKPOINT,
    "checkpoints/session3_steering_bank.pt": BANK_CHECKPOINT,
    "checkpoints/session3_router.pt": ROUTER_CHECKPOINT,
    "calibration/fixed_route_p99_norms.json": NORM_FILE,
    "calibration/candidate_norms_p99.json": CANDIDATE_NORMS,
    "selection/session3_static_bank_action.json": STATIC_ACTION,
    "training/alpha_calibration.json": ALPHA_REPORT,
}
missing = [name for name, path in required.items() if not path.is_file()]
bank_validation_errors = []
bank_summary = None
if BANK_CHECKPOINT.is_file():
    try:
        bank_state = torch.load(BANK_CHECKPOINT, map_location="cpu", weights_only=True)
        if bank_state.get("format") != "ars.independent_route_expert_bank" or bank_state.get("format_version") != 1:
            raise ValueError("unsupported bank checkpoint format")
        if not bank_state.get("training_complete") or not bank_state.get("route_experts_independent"):
            raise ValueError("bank is incomplete or route expert weights are shared")
        if list(bank_state.get("candidate_layers", [])) != list(CANDIDATE_LAYERS):
            raise ValueError("bank candidate layers differ from the handoff")
        if bank_state.get("architecture_id") != "ars.independent_route_low_rank_residual_bank" or bank_state.get("architecture_version") != "1.0.0":
            raise ValueError("unsupported bank architecture contract")
        bank_routes = normalize_route_catalog(bank_state.get("route_catalog", []), CANDIDATE_LAYERS)
        if bank_routes != tuple(tuple(route) for route in bank_state["route_catalog"]):
            raise ValueError("bank route catalog is not canonical")
        validate_route_expert_states(bank_routes, bank_state.get("route_experts", {}))
        norm_refs = bank_state.get("norm_refs", {})
        if set(norm_refs) != {str(layer) for layer in CANDIDATE_LAYERS}:
            raise ValueError("bank norm references do not match candidate layers")
        if any(not math.isfinite(float(value)) or float(value) <= 0 for value in norm_refs.values()):
            raise ValueError("bank contains invalid norm references")
        bank_summary = {
            "format": bank_state["format"],
            "format_version": bank_state["format_version"],
            "architecture_id": bank_state["architecture_id"],
            "architecture_version": bank_state["architecture_version"],
            "expert_architecture_id": bank_state.get("expert_architecture_id"),
            "expert_architecture_version": bank_state.get("expert_architecture_version"),
            "route_catalog": bank_state["route_catalog"],
            "router_catalog": bank_state.get("curriculum", {}).get("router_catalog", []),
            "route_expert_count": len(bank_state["route_experts"]),
            "route_experts_independent": True,
            "route_expert_parameter_count": bank_state["route_expert_parameter_count"],
            "route_spacing": bank_state.get("curriculum", {}).get("route_spacing", {}),
        }
    except Exception as exc:
        bank_validation_errors.append(f"{type(exc).__name__}: {exc}")
else:
    bank_validation_errors.append("missing steering-bank checkpoint")

code_files = {name: RUN_DIR / name for name in HANDOFF_MANIFEST.get("code", {})}
missing_code = [
    name for name, path in code_files.items()
    if not path.is_file() or sha256_file(path) != HANDOFF_MANIFEST["code"][name]["sha256"]
]
required_phases = (
    "runtime", "tests", "norms", "collection", "training",
    "candidate_norms", "alpha_calibration", "steering_bank", "router",
)
failed_phases = {
    name: PHASE_STATUS.get(name, "not_run")
    for name in required_phases
    if PHASE_STATUS.get(name) not in (
        {"passed", "warning_single_gpu"} if name == "runtime"
        else {"passed", "completed"} if name == "collection"
        else {"passed"}
    )
}
training_ready = not missing and not missing_code and not failed_phases and not bank_validation_errors
if training_ready:
    PHASE_STATUS["packaging"] = "passed"
else:
    PHASE_STATUS["packaging"] = (
        f"partial: missing={missing + missing_code}, "
        f"phases={failed_phases}, bank_validation_errors={bank_validation_errors}"
    )

manifest = {
    "format": "ars_training_bundle_v1",
    "training_ready": training_ready,
    "architecture_id": HANDOFF_MANIFEST["architecture_id"],
    "architecture_version": HANDOFF_MANIFEST["architecture_version"],
    "steering_bank": bank_summary,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "run_id": RUN_ID,
    "code_commit": HANDOFF_MANIFEST["code_commit"],
    "parent_handoff_bundle_sha256": sha256_file(FIXED_ADAPTER_INPUT) if HANDOFF_INPUT_TYPE == "zip" else None,
    "parent_handoff_content_sha256": HANDOFF_CONTENT_SHA256,
    "parent_handoff_input_type": HANDOFF_INPUT_TYPE,
    "model": HANDOFF_MANIFEST["model"],
    "route": FIXED_ROUTE,
    "candidate_layers": CANDIDATE_LAYERS,
    "rank": HANDOFF_MANIFEST["rank"],
    "rho_max": HANDOFF_MANIFEST["rho_max"],
    "norm_stat": HANDOFF_MANIFEST["norm_stat"],
    "seed": HANDOFF_MANIFEST["seed"],
    "shot_weights": HANDOFF_MANIFEST["shot_weights"],
    "training_profile": TRAINING_PROFILE_CONFIG,
    "selected_alpha": SELECTED_ALPHA,
    "alpha_selection": ALPHA_SELECTION,
    "phase_status": PHASE_STATUS,
    "required_phase_status": {name: PHASE_STATUS.get(name) for name in required_phases},
    "missing_required": missing + missing_code,
    "bank_validation_errors": bank_validation_errors,
    "artifacts": {
        name: {"sha256": sha256_file(path)}
        for name, path in required.items() if path.is_file()
    },
    "code": {
        name: {"sha256": HANDOFF_MANIFEST["code"][name]["sha256"]}
        for name in code_files if name in HANDOFF_MANIFEST.get("code", {})
    },
}
manifest_path = RUN_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
status_path = RUN_DIR / "phase_status.json"
status_path.write_text(json.dumps(PHASE_STATUS, indent=2) + "\n", encoding="utf-8")
final_bundle = RUN_DIR / "ars_training_bundle.zip"
with zipfile.ZipFile(final_bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(manifest_path, "manifest.json")
    archive.write(status_path, "training/phase_status.json")
    for name, path in required.items():
        if path.is_file():
            archive.write(path, name)
    for name, path in code_files.items():
        if path.is_file():
            archive.write(path, name)
    for relative in ("training/collection_plan.json", "training/collection_summary.json"):
        path = RUN_DIR / relative
        if path.is_file():
            archive.write(path, relative)
    if RFT_DIR.exists():
        for path in sorted(RFT_DIR.rglob("*")):
            if path.is_file():
                archive.write(path, f"training/rollouts/{path.name}")
    for log_name in ("alpha_calibration.log", "bank_training.log", "router_training.log"):
        path = RUN_DIR / "logs" / log_name
        if path.is_file():
            archive.write(path, f"training/{log_name}")
print(json.dumps({
    "final_bundle": str(final_bundle),
    "training_ready": training_ready,
    "phase_status": PHASE_STATUS,
    "failed_phases": failed_phases,
    "bank_validation_errors": bank_validation_errors,
    "missing": missing + missing_code,
}, indent=2), flush=True)

## Run receipt

Notebook 01b should report `training_ready: true` before its ZIP is attached to evaluation notebooks. If either the bank or router stage fails, preserve the run output and do not use its final ZIP as a completed training bundle.